# Acolyte – find SA jobs (Colab)

Runs the same job search as the daily GitHub Action, on demand:

1. **Gemini + Google Search** finds current South African postings (each link is verified).
2. Jobs are saved to your **Neon** database and classified.
3. Optional: copy the jobs into a **Google Sheet**.

The website updates at the next daily refresh, or straight away if you run *Refresh Automation Index* in GitHub Actions.

**First time:** click the 🔑 *Secrets* icon in the left sidebar and add `DATABASE_URL` and `GEMINI_API_KEY` (free at aistudio.google.com/apikey), with notebook access switched on. Optional: `ANTHROPIC_API_KEY`, `ADZUNA_APP_ID`, `ADZUNA_APP_KEY`.

In [ ]:
# 1. Get the code and install packages
!git clone -q --depth 1 https://github.com/dumichanda/acolyte-site.git 2>/dev/null || git -C acolyte-site pull -q
%cd acolyte-site
!pip install -q -r requirements.txt

In [ ]:
# 2. Load keys from Colab Secrets (never paste keys into the notebook)
import os
from google.colab import userdata
for name in ['DATABASE_URL', 'GEMINI_API_KEY', 'ANTHROPIC_API_KEY', 'ADZUNA_APP_ID', 'ADZUNA_APP_KEY']:
    try:
        os.environ[name] = userdata.get(name)
        print(f'{name}: loaded')
    except Exception:
        print(f'{name}: not set (skipped)')

In [ ]:
# 3. Search for jobs. ROLES_PER_RUN: 5 rotates through the list; 12 = every role now.
ROLES_PER_RUN = 5
!python pipeline/gemini_jobs.py --terms {ROLES_PER_RUN}
# Single role instead:  !python pipeline/gemini_jobs.py --term "claims administrator"

In [ ]:
# 4. Classify new jobs (automation score, role family, Acolyte service)
!python pipeline/classify.py

In [ ]:
# 5. See what's in the database
import sys; sys.path.insert(0, 'pipeline')
import pandas as pd, db
with db.connect() as conn:
    by_source = pd.read_sql('SELECT source, count(*) AS jobs, max(fetched_at)::date AS last_added FROM jobs GROUP BY 1 ORDER BY 2 DESC', conn)
    recent = pd.read_sql('''SELECT title, company, city, province, role_family, automation_score, url, posted_at::date AS posted, source
                           FROM v_jobs ORDER BY fetched_at DESC LIMIT 500''', conn)
display(by_source)
recent.head(20)

### Optional: copy jobs to Google Sheets
Creates (or updates) a sheet called **Acolyte – SA Jobs** in your Google Drive. You'll be asked to allow access the first time.

In [ ]:
# 6. Write the 500 most recent jobs to a Google Sheet
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
NAME = 'Acolyte – SA Jobs'
try:
    sh = gc.open(NAME)
except gspread.SpreadsheetNotFound:
    sh = gc.create(NAME)
ws = sh.sheet1
ws.clear()
rows = recent.astype(str).values.tolist()
ws.update([recent.columns.tolist()] + rows)
print('Done:', sh.url)